In [1]:
# import wavelet2DT_GPU as wt
# import wavelet2DT_GPU as wt
# import Wavelet2D1T as wt_angle
%matplotlib ipympl
import numpy as np
import scipy
import matplotlib.pyplot as plt
from itertools import product
from tqdm.notebook import tqdm
from matplotlib.colors import Normalize, LogNorm
from matplotlib.animation import FuncAnimation
import image as img
import cwt2D1T as cwt

In [ ]:
import os
comp_fits_dir = "../data/fits_20120327_morton/"
comp_fits_list = os.listdir(comp_fits_dir)
already_crop_data = False

for i, comp_fits in enumerate(comp_fits_list):
    from astropy.io import fits
    hdul = fits.open(comp_fits_dir+comp_fits)
    obstime = hdul[0].header['DATE-OBS']+' '+hdul[0].header['TIME-OBS']
    if i == 0:
        print(hdul[0].header['CDELT1'])
    doppler_map = hdul[3].data
    if not already_crop_data:
        sequence_doppler_map = np.copy(doppler_map[530:590, 250:370])
        already_crop_data = True
    else:
        sequence_doppler_map = np.dstack([sequence_doppler_map, doppler_map[530:590, 250:370]])

comp_data = np.transpose(sequence_doppler_map, (0,1,2))[:, :, 35:170]

dx = 3.286
dy = 3.286
dt = 30

计算时间轴上的傅里叶变换，查看功率谱

In [ ]:
fft_data = np.fft.fftn(comp_data, axes=[-1])
psd = np.mean(np.abs(fft_data[10:30, 20:40, :])**2, axis=(0, 1))/comp_data.shape[-1]*30
freq = np.fft.fftfreq(comp_data.shape[-1], 30)
positive_freq_mask = freq > 0
freq_pos = freq[positive_freq_mask]
psd_pos = psd[positive_freq_mask]

fig, ax = plt.subplots()
ax.plot(freq_pos, psd_pos)
ax.set_xlabel('Frequency (Hz)')
fig.savefig('./fig/傅里叶变换功率谱.png', dpi=300)

In [ ]:
plt.close(fig)

使用高斯滤波器进行傅里叶滤波

In [ ]:
f0 = 3.5e-3
delta_f = 0.4e-3
gaussian_filter = np.exp(-(freq - f0)**2 / delta_f**2)[None, None, :]
fft_data_filtered = gaussian_filter * fft_data
data_filtered = np.fft.ifftn(fft_data_filtered, axes=[-1]).real

In [ ]:
# data_filtered = comp_data
fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(data_filtered[:, :, 0], cmap='gray', origin='lower', vmin=-0.5, vmax=0.5)
ax.set_aspect('equal')
title = ax.set_title(f'comp data filtered, frame {0:03d}')
plt.colorbar(im, ax=ax)
def update(frame):
    im.set_data(data_filtered[:, :, frame])
    title.set_text(f'comp data filtered, frame {frame:03d}')
    return [im, title]
ani = FuncAnimation(fig, update, frames=data_filtered.shape[-1], interval=100, blit=True)
ani.save('./movie/comp数据滤波.mp4', dpi=300, fps=10)

In [ ]:
plt.close(fig)